# 📖 Marker OCR Runner — Free T4 GPU

This notebook runs [Marker](https://github.com/VikParuchuri/marker) on a **free Colab T4 GPU** to OCR scanned PDFs.

## Why use this?
- Marker on M3 MacBook CPU: **3+ hours** for a 600-page book (and thermal throttling)
- Marker on T4 GPU here: **~15-35 minutes** for the same book

## Before you start
1. **Set GPU runtime**: `Runtime` → `Change runtime type` → **T4 GPU**
2. Upload your scanned PDF to Google Drive (or use the upload cell below)

## How it works
1. Installs `marker-pdf` and dependencies
2. Mounts your Google Drive for input/output
3. Processes the PDF in **chunks** (100 pages each) — if the session disconnects, re-run and it resumes from the last completed chunk
4. Merges all chunk outputs into a single JSON file
5. Output stays in Google Drive — download to your Mac, then run `parse_marker_output.py` locally

## Output
After this notebook completes, you'll have:
- `marker_output/<book_name>/merged/<book_name>.json` — the merged Marker JSON
- `marker_output/<book_name>/merged/images/` — all extracted images

Copy these to your local machine and run:
```bash
python -m preprocessing.parse_marker_output \
    <path/to/merged.json> \
    <path/to/merged/images> \
    books/processed/<book_name> \
    "Book Title"
```

---
## Cell 1 — Verify GPU Runtime

**Run this first!** If it says "CPU only", go to `Runtime` → `Change runtime type` → `T4 GPU` and re-run.

In [5]:
# ────────────────────────────────────────────────────────────────────────
# Cell 1: Verify that a GPU is available.
#
# If this prints "CPU only", you need to switch runtimes:
#   Runtime → Change runtime type → T4 GPU
# Colab's free tier gives you a T4 (16GB VRAM), which is more than
# enough for Marker's vision-transformer models.
# ────────────────────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"✅ GPU detected: {gpu_name} ({gpu_mem:.1f} GB VRAM)")
    print(f"   CUDA version: {torch.version.cuda}")
else:
    print("❌ No GPU detected — running on CPU only.")
    print("   Go to Runtime → Change runtime type → T4 GPU")
    print("   Then re-run this cell.")

ModuleNotFoundError: No module named 'torch'

---
## Cell 2 — Install Marker

In [4]:
# ────────────────────────────────────────────────────────────────────────
# Cell 2: Install marker-pdf.
#
# This pulls in torch, surya-ocr, transformers, etc.
# First run also downloads model weights (~1-2 GB) — one-time cost.
# Subsequent runs in the same session skip the download.
# ────────────────────────────────────────────────────────────────────────
!pip install -q marker-pdf

# Also install PyMuPDF for page-count detection
!pip install -q pymupdf

# Verify marker_single is on PATH
!which marker_single && echo '✅ marker_single found' || echo '❌ marker_single not found'

zsh:1: /Users/gufran/Desktop/EngineerGPT/.venv/bin/pip: bad interpreter: /Users/gufran/Desktop/pustak/.venv/bin/python3: no such file or directory

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
zsh:1: /Users/gufran/Desktop/EngineerGPT/.venv/bin/pip: bad interpreter: /Users/gufran/Desktop/pustak/.venv/bin/python3: no such file or directory

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
/Users/gufran/.pyenv/shims/marker_single
✅ marker_single found


---
## Cell 3 — Mount Google Drive

This gives the notebook access to your Drive for input (PDF) and output (Marker JSON + images).

You'll be prompted to authorize access — click through the Google auth flow.

In [ ]:
# ────────────────────────────────────────────────────────────────────────
# Cell 3: Mount Google Drive.
#
# After mounting, your Drive appears at /content/drive/MyDrive/
# We use a dedicated folder for this project's files:
#   /content/drive/MyDrive/engineering-rag-assistant/
#     books/
#       raw/          ← put your scanned PDFs here
#     marker_output/  ← chunked + merged output lands here
# ────────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Create the project folder structure in Drive (idempotent)
import os
BASE_DIR = "/content/drive/MyDrive/engineering-rag-assistant"
os.makedirs(f"{BASE_DIR}/books/raw", exist_ok=True)
os.makedirs(f"{BASE_DIR}/marker_output", exist_ok=True)
print(f"\n✅ Project folder ready at: {BASE_DIR}")
print(f"   Upload PDFs to: {BASE_DIR}/books/raw/")

---
## Cell 4 — Force GPU Device

Explicitly tell Marker/PyTorch to use CUDA. Don't rely on auto-detection.

In [ ]:
# ────────────────────────────────────────────────────────────────────────
# Cell 4: Force GPU device.
#
# Marker uses the TORCH_DEVICE env var to decide where to run inference.
# We set it explicitly to "cuda" so it doesn't fall back to CPU.
# ────────────────────────────────────────────────────────────────────────
import os
os.environ["TORCH_DEVICE"] = "cuda"
print("✅ TORCH_DEVICE set to 'cuda'")

---
## Cell 5 — Configure Input/Output Paths

**Edit the `BOOK_NAME` variable below** to match your PDF filename (without `.pdf`).

For example, if your PDF is `fluid_mechanics.pdf`, set `BOOK_NAME = "fluid_mechanics"`.

In [ ]:
# ────────────────────────────────────────────────────────────────────────
# Cell 5: Configure which book to process.
#
# ⚠️  EDIT THIS to match your PDF filename (without the .pdf extension).
#     The PDF must already be in Google Drive at:
#     /content/drive/MyDrive/engineering-rag-assistant/books/raw/<name>.pdf
# ────────────────────────────────────────────────────────────────────────

BOOK_NAME = "your_book_name_here"   # ← EDIT THIS (e.g. "fluid_mechanics")

# Whether to force full re-OCR on every page.
# Set to True for pure image-only scans (0 embedded text).
# Set to False for scans with a partial text layer (faster — Marker
# uses the existing text where it can, only re-OCR'ing where needed).
FORCE_OCR = True

# Pages per chunk.  GPU is fast, so we can do bigger chunks than CPU.
# 100 is a good default.  Increase to 200 if you're confident the
# session won't disconnect (reduces subprocess overhead).
CHUNK_SIZE = 100

# Timeout per chunk in seconds.  GPU should finish 100 pages well
# within 30 minutes.  If a chunk exceeds this, it's logged and skipped.
CHUNK_TIMEOUT = 1800

# ── Derived paths (don't edit these) ──────────────────────────────────
BASE_DIR = "/content/drive/MyDrive/engineering-rag-assistant"
INPUT_PDF = f"{BASE_DIR}/books/raw/{BOOK_NAME}.pdf"
OUTPUT_DIR = f"{BASE_DIR}/marker_output/{BOOK_NAME}"

# Verify the PDF exists
import os
if os.path.exists(INPUT_PDF):
    size_mb = os.path.getsize(INPUT_PDF) / (1024 * 1024)
    print(f"✅ Found PDF: {INPUT_PDF} ({size_mb:.1f} MB)")
else:
    print(f"❌ PDF not found: {INPUT_PDF}")
    print(f"   Upload your PDF to Google Drive at:")
    print(f"   {BASE_DIR}/books/raw/{BOOK_NAME}.pdf")

print(f"\n📁 Output will be saved to: {OUTPUT_DIR}")
print(f"   Force OCR: {FORCE_OCR}")
print(f"   Chunk size: {CHUNK_SIZE} pages")
print(f"   Chunk timeout: {CHUNK_TIMEOUT}s")

---
## Cell 6 — Run Marker OCR (Chunked, with Resume Support)

This is the main processing cell.  It:
1. Gets the PDF page count
2. Splits into chunks of `CHUNK_SIZE` pages
3. **Skips chunks that already have output** (resume after disconnect!)
4. Runs `marker_single` on each remaining chunk
5. Logs progress so you can see which chunk is running

If the session disconnects mid-run, just re-run this cell — it picks up where it left off.

In [ ]:
# ────────────────────────────────────────────────────────────────────────
# Cell 6: Run Marker OCR in chunks with resume support.
#
# This is the same chunking logic as the local marker_ocr.py, adapted
# for the Colab environment.  Key differences from the local version:
#   - Larger default chunk size (100 vs 50) because GPU is faster
#   - Output goes to Google Drive for persistence across sessions
#   - Same resume logic: if a chunk's output dir already has a .json
#     file, it's skipped (completed in a previous run)
# ────────────────────────────────────────────────────────────────────────
import subprocess
import time
import json
import os
from pathlib import Path

import pymupdf  # PyMuPDF — for page count

# ── Get total page count ──────────────────────────────────────────────
doc = pymupdf.open(INPUT_PDF)
total_pages = len(doc)
doc.close()
print(f"📄 Total pages: {total_pages}")

# ── Compute chunks ────────────────────────────────────────────────────
chunks = []
for start in range(0, total_pages, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE - 1, total_pages - 1)
    chunks.append((start, end))

print(f"📦 Split into {len(chunks)} chunks of ~{CHUNK_SIZE} pages each")

# ── Create output directories ────────────────────────────────────────
chunks_dir = f"{OUTPUT_DIR}/chunks"
os.makedirs(chunks_dir, exist_ok=True)

# ── Process each chunk ────────────────────────────────────────────────
succeeded = 0
failed = 0
skipped = 0
failed_ranges = []
total_start = time.time()

for i, (start, end) in enumerate(chunks, 1):
    chunk_dir = f"{chunks_dir}/chunk_{start:04d}_{end:04d}"
    
    # ── Resume support: check if this chunk already has output ────
    # A chunk is "done" if its directory exists and contains a .json file.
    chunk_path = Path(chunk_dir)
    if chunk_path.exists() and list(chunk_path.rglob("*.json")):
        print(f"  ⏭️  Chunk {i}/{len(chunks)} (pages {start}–{end}): "
              f"CACHED ✓ (skipping)")
        skipped += 1
        continue

    # ── Build the marker_single command ───────────────────────────
    os.makedirs(chunk_dir, exist_ok=True)
    page_range = f"{start}-{end}"

    cmd = [
        "marker_single", INPUT_PDF,
        "--output_format", "json",
        "--output_dir", chunk_dir,
        "--page_range", page_range,
    ]
    if FORCE_OCR:
        cmd.append("--force_ocr")

    # ── Run ───────────────────────────────────────────────────────
    print(f"  🔄 Chunk {i}/{len(chunks)} (pages {start}–{end}): "
          f"processing...")
    chunk_start = time.time()

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=CHUNK_TIMEOUT,
        )
        elapsed = time.time() - chunk_start

        if result.returncode == 0:
            pages_per_sec = (end - start + 1) / elapsed if elapsed > 0 else 0
            print(f"  ✅ Chunk {i}/{len(chunks)} (pages {start}–{end}): "
                  f"DONE in {elapsed:.1f}s ({pages_per_sec:.1f} pages/sec)")
            succeeded += 1
        else:
            stderr_preview = result.stderr[:500] if result.stderr else "(no stderr)"
            print(f"  ❌ Chunk {i}/{len(chunks)} (pages {start}–{end}): "
                  f"FAILED (exit code {result.returncode})")
            print(f"     {stderr_preview}")
            failed += 1
            failed_ranges.append(f"{start}-{end}")

    except subprocess.TimeoutExpired:
        elapsed = time.time() - chunk_start
        print(f"  ⏰ Chunk {i}/{len(chunks)} (pages {start}–{end}): "
              f"TIMEOUT after {elapsed:.1f}s — skipping")
        failed += 1
        failed_ranges.append(f"{start}-{end} (timeout)")

    except FileNotFoundError:
        print(f"  ❌ marker_single not found! Run Cell 2 to install.")
        break

total_elapsed = time.time() - total_start

# ── Summary ───────────────────────────────────────────────────────────
print(f"\n{'═' * 60}")
print(f"  OCR Processing Summary")
print(f"{'═' * 60}")
print(f"  Book         : {BOOK_NAME}")
print(f"  Total pages  : {total_pages}")
print(f"  Total chunks : {len(chunks)}")
print(f"  Succeeded    : {succeeded} (newly processed)")
print(f"  Cached       : {skipped} (from previous run)")
print(f"  Failed       : {failed}")
if failed_ranges:
    print(f"  Failed pages : {', '.join(failed_ranges)}")
print(f"  Total time   : {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)")
print(f"{'═' * 60}")

---
## Cell 7 — Merge Chunk Outputs

Combines all per-chunk JSON files into a single merged JSON that `parse_marker_output.py` can consume.

This is the same merge logic as the local `marker_ocr.py` — it concatenates the block lists from each chunk in page order and copies all images into one directory.

In [ ]:
# ────────────────────────────────────────────────────────────────────────
# Cell 7: Merge all chunk outputs into one combined JSON + images dir.
#
# This produces the exact same output format that the local marker_ocr.py
# merge step creates, so parse_marker_output.py can consume it without
# any changes.
# ────────────────────────────────────────────────────────────────────────
import json
import shutil
from pathlib import Path

chunks_dir = Path(f"{OUTPUT_DIR}/chunks")
merged_dir = Path(f"{OUTPUT_DIR}/merged")
merged_images_dir = merged_dir / "images"

# Create merged output directories
merged_dir.mkdir(parents=True, exist_ok=True)
merged_images_dir.mkdir(parents=True, exist_ok=True)

# Collect all completed chunk directories (sorted by name = page order)
chunk_dirs = sorted([
    d for d in chunks_dir.iterdir()
    if d.is_dir() and d.name.startswith("chunk_")
])

print(f"Found {len(chunk_dirs)} chunk directories")

# ── Merge JSON blocks ─────────────────────────────────────────────────
all_blocks = []

for chunk_dir in chunk_dirs:
    # Find the JSON file in this chunk's output
    json_files = list(chunk_dir.rglob("*.json"))
    if not json_files:
        print(f"  ⚠️  No JSON in {chunk_dir.name} — skipping")
        continue

    json_path = json_files[0]
    chunk_data = json.loads(json_path.read_text(encoding="utf-8"))

    # Extract blocks regardless of JSON structure
    # (same logic as marker_ocr.py's _merge_chunk_outputs)
    if isinstance(chunk_data, list):
        all_blocks.extend(chunk_data)
    elif isinstance(chunk_data, dict):
        if "children" in chunk_data:
            children = chunk_data["children"]
            if isinstance(children, list):
                all_blocks.extend(children)
            else:
                all_blocks.append(chunk_data)
        else:
            all_blocks.append(chunk_data)

    # Copy images from this chunk to the merged images directory
    for img_dir_candidate in chunk_dir.rglob("images"):
        if img_dir_candidate.is_dir():
            for img_file in img_dir_candidate.iterdir():
                if img_file.is_file() and not img_file.name.startswith("."):
                    dest = merged_images_dir / img_file.name
                    if dest.exists():
                        # Avoid overwrites — prefix with chunk name
                        dest = merged_images_dir / f"{chunk_dir.name}_{img_file.name}"
                    shutil.copy2(str(img_file), str(dest))

    print(f"  ✅ Merged {chunk_dir.name}")

# ── Write merged JSON ─────────────────────────────────────────────────
merged_json_path = merged_dir / f"{BOOK_NAME}.json"
merged_json_path.write_text(
    json.dumps(all_blocks, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# Count images
image_count = len(list(merged_images_dir.iterdir())) if merged_images_dir.exists() else 0

print(f"\n{'═' * 60}")
print(f"  Merge Complete")
print(f"{'═' * 60}")
print(f"  Merged JSON  : {merged_json_path}")
print(f"  Total blocks : {len(all_blocks)}")
print(f"  Total images : {image_count}")
print(f"  Images dir   : {merged_images_dir}")
print(f"{'═' * 60}")

---
## Cell 8 — Next Steps

The Marker OCR output is now in your Google Drive.  Here's what to do next:

### Option A: Google Drive for Desktop (recommended, zero friction)
If you have [Google Drive for Desktop](https://www.google.com/drive/download/) installed on your Mac, the output is already accessible as a local folder.  Just run:

```bash
# On your Mac — paths will vary based on your Drive mount point
python -m preprocessing.parse_marker_output \
    "/Volumes/GoogleDrive/My Drive/engineering-rag-assistant/marker_output/<book_name>/merged/<book_name>.json" \
    "/Volumes/GoogleDrive/My Drive/engineering-rag-assistant/marker_output/<book_name>/merged/images" \
    books/processed/<book_name> \
    "Book Title"
```

### Option B: Manual download
1. In the Colab file browser (left sidebar), navigate to the `merged/` folder
2. Right-click → Download (or use the cell below to create a zip)
3. Unzip on your Mac into `books/processed/<book_name>/`
4. Run `parse_marker_output.py` and then `build_index.py`

In [ ]:
# ────────────────────────────────────────────────────────────────────────
# Cell 8 (optional): Create a downloadable zip of the merged output.
#
# Use this if you prefer manual download instead of Google Drive sync.
# The zip file will appear in the Colab file browser — right-click to
# download it.
# ────────────────────────────────────────────────────────────────────────
import shutil

zip_path = f"{OUTPUT_DIR}/{BOOK_NAME}_marker_output"
shutil.make_archive(zip_path, 'zip', str(merged_dir))
print(f"✅ Zip created: {zip_path}.zip")
print(f"   Download this from the Colab file browser (left sidebar)")
print(f"   or from Google Drive.")